# Блок 7. Основы искусственного интеллекта и машинного обучения
# Занятие 4. Практика регрессии на датасете недвижимости

## Цель занятия

На прошлом занятии мы изучили линейную регрессию.

Сегодня будем работать как настоящий аналитик данных:

- использовать больше признаков;
- сравнивать качество модели;
- анализировать важность признаков;
- интерпретировать ошибки;
- строить графики;
- формировать бизнес-отчёт.

## Датасет

California Housing

https://raw.githubusercontent.com/ageron/handson-ml/master/datasets/housing/housing.csv

## Связь с дипломом

Это практически готовый шаблон для дипломов:

- недвижимость;
- продажи;
- финансы;
- медицина;
- спорт.


Сначала изучите SOLVED ноутбук, затем выполните TODO самостоятельно.

## Ячейка 1. Загрузка датасета

### Алгоритм
Загружаем реальные данные недвижимости.

In [11]:
# TODO: выполните самостоятельно задание ячейки 1
import pandas as pd
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')

file_path = Path("/content/drive/MyDrive/Базы/sales_data.xlsx")
df = pd.read_excel(file_path, sheet_name="sales_data", header=0)

print(df.shape)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
(3000, 8)


## Ячейка 2. Изучение признаков

### Алгоритм
Смотрим структуру данных.

In [12]:
# TODO: выполните самостоятельно задание ячейки 2
display(df.head())
print(df.columns.tolist())

,date,manager,city,category,product,quantity,price,discount
0,2024-01-01,Иванов А.С.,Екатеринбург,Одежда,Кроссовки,5,6191.15,0.35
1,2024-01-01,Сидоров К.Д.,Челябинск,Книги,Детектив,1,734.32,0.40
2,2024-01-01,Кузнецов П.Р.,Москва,Дом и сад,Увлажнитель воздуха,1,5343.56,0.27
3,2024-01-01,Иванов А.С.,Казань,Продукты,Шоколад,3,149.17,0.46
4,2024-01-01,Иванов А.С.,Москва,Автотовары,Ароматизатор,16,382.44,0.26


['date', 'manager', 'city', 'category', 'product', 'quantity', 'price', 'discount']


## Ячейка 3. Подготовка данных

### Алгоритм
Выбираем больше признаков.

In [13]:
# TODO: выполните самостоятельно задание ячейки 3
data=df[[
    "date",
    "product",
    "city",
    "manager",
    "quantity",
    "category",
    "price",
    "discount"
]].dropna()

X=data.drop(columns=["quantity"])
y=data["quantity"]
display(X.head())
display(y.head())

,date,product,city,manager,category,price,discount
0,2024-01-01,Кроссовки,Екатеринбург,Иванов А.С.,Одежда,6191.15,0.35
1,2024-01-01,Детектив,Челябинск,Сидоров К.Д.,Книги,734.32,0.40
2,2024-01-01,Увлажнитель воздуха,Москва,Кузнецов П.Р.,Дом и сад,5343.56,0.27
3,2024-01-01,Шоколад,Казань,Иванов А.С.,Продукты,149.17,0.46
4,2024-01-01,Ароматизатор,Москва,Иванов А.С.,Автотовары,382.44,0.26


,quantity
0,5
1,1
2,1
3,3
4,16


## Ячейка 4. Разделение данных

### Алгоритм
Train/Test Split.

In [14]:
# TODO: выполните самостоятельно задание ячейки 4
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
    )

## Ячейка 5. Масштабирование

### Алгоритм
Используем StandardScaler.

In [18]:
# TODO: выполните самостоятельно задание ячейки 5
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Комментарий: Убедимся, что столбец 'date' имеет тип datetime для корректного извлечения признаков.
X_train['date'] = pd.to_datetime(X_train['date'])
X_test['date'] = pd.to_datetime(X_test['date'])

# Комментарий: Извлекаем новые временные признаки из столбца 'date'.
# Это поможет модели учитывать сезонность и другие временные паттерны.
X_train['year'] = X_train['date'].dt.year
X_train['month'] = X_train['date'].dt.month
X_train['day_of_week'] = X_train['date'].dt.dayofweek
X_train['day_of_month'] = X_train['date'].dt.day
X_train['quarter'] = X_train['date'].dt.quarter

X_test['year'] = X_test['date'].dt.year
X_test['month'] = X_test['date'].dt.month
X_test['day_of_week'] = X_test['date'].dt.dayofweek
X_test['day_of_month'] = X_test['date'].dt.day
X_test['quarter'] = X_test['date'].dt.quarter

# Комментарий: После извлечения информативных признаков, исходный столбец 'date' удаляется.
# Он больше не нужен и не может быть обработан StandardScaler напрямую.
X_train_processed = X_train.drop(columns=['date'])
X_test_processed = X_test.drop(columns=['date'])

# Комментарий: Определяем категориальные столбцы, которые будут подвергнуты One-Hot кодированию.
categorical_cols = ['product', 'city', 'manager', 'category']

# Комментарий: Применяем One-Hot кодирование к категориальным признакам.
# Параметр drop_first=True помогает избежать проблемы мультиколлинеарности (dummy variable trap).
X_train_encoded = pd.get_dummies(X_train_processed, columns=categorical_cols, drop_first=True)
X_test_encoded = pd.get_dummies(X_test_processed, columns=categorical_cols, drop_first=True)

# Комментарий: Выравниваем столбцы обучающего и тестового наборов после One-Hot кодирования.
# Это критически важно для обеспечения того, чтобы оба набора имели одинаковые признаки в одном и том же порядке,
# даже если какая-либо категория присутствует только в одном из них.

# Определяем столбцы, которые есть в обучающем наборе, но отсутствуют в тестовом
missing_in_test = set(X_train_encoded.columns) - set(X_test_encoded.columns)
for c in missing_in_test:
    X_test_encoded[c] = 0 # Добавляем отсутствующие столбцы в тестовый набор со значениями 0

# Определяем столбцы, которые есть в тестовом наборе, но отсутствуют в обучающем
missing_in_train = set(X_test_encoded.columns) - set(X_train_encoded.columns)
for c in missing_in_train:
    X_train_encoded[c] = 0 # Добавляем отсутствующие столбцы в обучающий набор со значениями 0

# Комментарий: Убеждаемся, что порядок столбцов в тестовом наборе совпадает с порядком в обучающем наборе.
X_test_encoded = X_test_encoded[X_train_encoded.columns]

# Комментарий: Инициализируем StandardScaler.
# Он будет стандартизировать признаки, приводя их к нулевому среднему и единичному стандартному отклонению.
scaler = StandardScaler()

# Комментарий: Применяем масштабирование к обучающему набору данных.
# fit_transform() одновременно обучается на данных и преобразует их.
X_train_scaled = scaler.fit_transform(X_train_encoded)

# Комментарий: Применяем масштабирование к тестовому набору данных, используя параметры,
# полученные (обученные) на обучающем наборе. Использование только transform() предотвращает утечку данных.
X_test_scaled = scaler.transform(X_test_encoded)

удалил столбец 'date' перед масштабированием по нескольким причинам:

Требования StandardScaler: StandardScaler предназначен для масштабирования числовых признаков, которые имеют непрерывное распределение. Он вычисляет среднее значение и стандартное отклонение для каждого столбца и использует их для преобразования данных. Столбец 'date' (даже если его преобразовать в числовое представление, например, в отметку времени Unix) не является обычным числовым признаком в том смысле, в котором его обычно масштабируют.

Извлечение временных признаков (Feature Engineering): Для таких признаков, как дата, часто гораздо эффективнее извлекать из них новые, более информативные числовые признаки, которые затем можно масштабировать. Например, можно извлечь:

год
месяц
день недели (показывает, является ли это будним днем или выходным)
день месяца
квартал
праздник (бинарный признак, является ли дата праздником)
Эти новые признаки лучше отражают временные паттерны и сезонность, которые могут быть важны для вашей модели, чем простое числовое представление даты. Если бы мы просто преобразовали дату в число и отмасштабировали, это могло бы привести к потере ценной информации о цикличности или сезонности.

Причина ошибки: Основная причина, по которой пришлось удалить столбец 'date' на данном этапе, заключалась в том, что StandardScaler выдал бы ошибку, если бы ему передали нес числовые данные или данные, которые он не может корректно интерпретировать как числовые для масштабирования. Чтобы избежать ошибки и продолжить работу с числовыми и категориальными признаками, я временно исключил 'date'.

Если временные паттерны важны для предсказания quantity, то следующим шагом было бы создание новых признаков из столбца date (как описано в пункте 2) перед их масштабированием и обучением модели.

## Ячейка 6. Обучение модели

### Алгоритм
Строим LinearRegression.

In [ ]:
# TODO: выполните самостоятельно задание ячейки 6

## Ячейка 7. Важность признаков

### Алгоритм
Анализируем коэффициенты.

In [ ]:
# TODO: выполните самостоятельно задание ячейки 7

## Ячейка 8. Метрики качества

### Алгоритм
Считаем MAE RMSE R2.

In [ ]:
# TODO: выполните самостоятельно задание ячейки 8

## Ячейка 9. Визуализация

### Алгоритм
Строим графики качества.

In [ ]:
# TODO: выполните самостоятельно задание ячейки 9

## Ячейка 10. Мини-проект

### Алгоритм
Формируем бизнес-отчет.

In [ ]:
# TODO: выполните самостоятельно задание ячейки 10